# P05 Модель как сервис

**Эталон решения**. Полное объяснение находится в Pages. Этот файл содержит исполняемый практикум.

## Цель

Выделить границу API, версию модели и обработку некорректного запроса.

**Перед началом:** P01–P03; знакомые HTTP и JSON. Новая тема — отдельный жизненный цикл параметров модели.

## Подготовка

Запускайте из корня репозитория в его виртуальном окружении. Первичная установка зависимостей описана в README. После установки выполнение не использует сеть, платные API или GPU.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mlcourse" / "pilot.py").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("Откройте Jupyter из корня ml-llm-course")
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from mlcourse.pilot import (
    mse, predict_line, linear_gradient, central_difference, fit_standardizer,
    matvec, fit_gradient, make_dataset, run_experiment, predict_payload,
)
plt.rcParams.update({"figure.figsize": (7.2, 4.2), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
FIGURES = ROOT / "artifacts" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Данные: искусственные учебные примеры. CPU. Состояние ученика не оценивается автоматически.")

## Разобранный пример

Обучение может выполняться отдельно от сервиса. Сервис получает параметры конкретной версии и применяет их. Сохранение истории запросов — изменение состояния приложения, но не обязательно обучение. Граница API проверяет вход до численного вычисления.

### Проверить контракт предсказания

In [ ]:
request = {"x": 4}
response = predict_payload(request)
print(response)
assert response == {"prediction": 9., "model_version": "v1"}

### Проверить невалидные входы

In [ ]:
for bad in [{}, {"x": "4"}, {"x": float("nan")}, {"x": True}]:
    try:
        predict_payload(bad)
    except ValueError as error:
        print(type(bad).__name__, "→ отклонён:", error)
    else:
        raise AssertionError("Некорректный запрос принят")

## Практика с поддержкой

Нарисуйте запрос → валидация → параметры v1 → предсказание → ответ. Добавьте место хранения версии и метрики ошибок.

## Самостоятельная задача

Составьте контракт POST /predict и минимум три негативных сценария. Затем опишите выпуск v2 и возврат на v1. Для проверки функции можно использовать tools/check_submission.py <файл> --task predict. Исполняемый HTTP-обработчик есть в mlcourse.pilot.prediction_handler; интеграционный тест запускает его только на 127.0.0.1.

Сначала выполните свою попытку. Подсказки и разбор доступны у наставника; использованную помощь отмечайте.

### Эталон

Открывайте после собственной попытки. После разбора понадобится новый вариант для проверки самостоятельности.

In [ ]:
contract = {
    "method": "POST", "path": "/predict", "input": {"x": 4},
    "success": {"prediction": 9., "model_version": "v1"},
    "invalid": "HTTP 400: отсутствует x, строка вместо числа, NaN/Infinity",
    "release": "Проверить v2 на отдельном наборе; сохранить v1; переключить версию; наблюдать ошибки.",
    "rollback": "Вернуть прежние параметры и запись версии; не переобучать во время отката."
}
print(json.dumps(contract, ensure_ascii=False, indent=2))

## Проверка

Контракт совпадает с кодом; версия возвращается; обучение отделено от обработки запроса; описаны проверка и откат.

Успешный Run All подтверждает запуск подготовленных примеров. Самостоятельная работа и понимание требуют отдельной проверки наставником.

## Следующий шаг

Что произойдёт, если код нормализации обновили, а параметры модели оставили прежними?

[Основной источник](https://docs.python.org/3/library/http.server.html). Все значения в примерах синтетические. Сохраняйте исходные ноутбуки; для своей работы используйте копию.